# FactIndic interactive summary and evidence check

Paste one **Roman-script Hinglish** article or passage. This live Colab demo generates an English summary, splits it into factual claims, and shows each claim beside source evidence.

## Run it

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Run all cells from the top. The first run downloads public model weights.
3. Paste Roman-script Hinglish, select a Qwen3 size and prompt, then click **Summarize and check claims**.
4. Review the claim table and quoted evidence. Use the button to download a JSON report.

This is an **AI-assisted research demo**, not a factuality certificate. The selected Qwen3 model generates and checks its own summary. The code verifies that each accepted evidence quote appears in the exact source text shown to the model, but the model’s support/contradiction judgment can still be wrong. Human validation was skipped for this project version.

The fixed 30-source, 120-summary comparison remains in [`factindic_final_open_hinglish_colab.ipynb`](https://colab.research.google.com/github/Sirohi0/factindic/blob/main/factindic_final_open_hinglish_colab.ipynb). This separate notebook is the live single-input demonstration. Inputs and results stay in the Colab runtime unless you download the report; the notebook does not write them to Drive or the public repository.


## 1. Install the inference tools

Use a GPU runtime before running this cell. It downloads models for inference and does not train them.

In [ ]:
%pip -q install "transformers>=4.51.3,<5" accelerate sentencepiece ipywidgets


## 2. Initialize the GPU runtime

In [ ]:
import gc
import hashlib
import json
import re
import time
import unicodedata
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import torch
from IPython.display import clear_output, display, Markdown
from google.colab import files, output
from transformers import AutoModelForCausalLM, AutoTokenizer

output.enable_custom_widget_manager()
assert torch.cuda.is_available(), "Choose Runtime → Change runtime type → T4 GPU, then rerun this cell."
print("GPU:", torch.cuda.get_device_name(0))

MODEL_IDS = {
    "Qwen3 1.7B": "Qwen/Qwen3-1.7B",
    "Qwen3 4B": "Qwen/Qwen3-4B",
}
MAX_SOURCE_TOKENS = 2048
SUMMARY_TOKEN_LIMIT = 192
CLAIM_CHECK_TOKEN_LIMIT = 768
_loaded_model_key = None
_model = None
_tokenizer = None
_last_result = None

def load_model(model_key):
    global _loaded_model_key, _model, _tokenizer
    if _loaded_model_key == model_key and _model is not None:
        return _model, _tokenizer
    if _model is not None:
        del _model, _tokenizer
        _model, _tokenizer = None, None
        gc.collect()
        torch.cuda.empty_cache()
    model_id = MODEL_IDS[model_key]
    print(f"Loading {model_id} …")
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto"
    ).eval()
    _loaded_model_key = model_key
    return _model, _tokenizer

def chat_generate(model, tokenizer, user_prompt, max_new_tokens):
    messages = [{"role": "user", "content": user_prompt}]
    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", enable_thinking=False
    ).to(model.device)
    input_tokens = int(encoded["input_ids"].shape[-1])
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **encoded, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
            repetition_penalty=1.03, no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
    elapsed = time.perf_counter() - started
    new_ids = output_ids[0][input_tokens:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip(), int(len(new_ids)), elapsed

def normalize_quote(text):
    return " ".join(unicodedata.normalize("NFKC", str(text)).split()).casefold()

def parse_json_object(text):
    text = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", text.strip(), flags=re.I)
    start = text.find("{")
    if start < 0:
        raise ValueError("The model did not return a JSON object.")
    parsed, _ = json.JSONDecoder().raw_decode(text[start:])
    return parsed

def prepare_source(source, tokenizer):
    source_ids = tokenizer.encode(source, add_special_tokens=False)
    truncated = len(source_ids) > MAX_SOURCE_TOKENS
    source_used = tokenizer.decode(source_ids[:MAX_SOURCE_TOKENS], skip_special_tokens=True)
    return source_used, len(source_ids), truncated

def make_summary(source_used, prompt_style, model, tokenizer):
    if prompt_style == "Standard":
        instruction = (
            "Summarize this Roman-script Hinglish news text in clear, concise English. "
            "Preserve the central event and stated people, places, numbers, and outcomes. "
            "Use only information explicitly present in the source."
        )
    else:
        instruction = (
            "Write a concise English summary of this Roman-script Hinglish text. "
            "Use only facts explicitly stated in the source. Do not infer causes, intentions, "
            "dates, quantities, identities, or outcomes. Omit unclear details."
        )
    prompt = f"{instruction}\n\nSource text:\n{source_used}\n\nEnglish summary:"
    summary, tokens, seconds = chat_generate(model, tokenizer, prompt, SUMMARY_TOKEN_LIMIT)
    return summary, tokens, seconds, tokens >= SUMMARY_TOKEN_LIMIT

def check_claims(source_used, summary, model, tokenizer):
    prompt = (
        "You are checking whether a summary is supported by its source. Split it into up to "
        "12 short, atomic factual claims. For each claim choose exactly one status: supported "
        "(source explicitly supports it), contradicted (source explicitly says something "
        "incompatible), not_found (source does not state it), or unclear (source wording does "
        "not let you decide). For supported or contradicted claims, give one short evidence "
        "quote copied exactly from the source. For not_found or unclear claims, use an empty "
        "evidence string. Never invent or rewrite a quote. Return JSON only in this shape: "
        '{"claims":[{"claim":"...","status":"supported|contradicted|not_found|unclear",'
        '"evidence":"exact source quote or empty"}]}\n\n'
        f"SOURCE:\n{source_used}\n\nSUMMARY:\n{summary}\n"
    )
    raw, tokens, seconds = chat_generate(model, tokenizer, prompt, CLAIM_CHECK_TOKEN_LIMIT)
    parsed = parse_json_object(raw)
    raw_claims = parsed.get("claims", [])
    if not isinstance(raw_claims, list) or not raw_claims:
        raise ValueError("No claim list returned. Rerun or inspect the summary manually.")
    allowed = {"supported", "contradicted", "not_found", "unclear"}
    normalized_source = normalize_quote(source_used)
    results = []
    for item in raw_claims[:12]:
        claim = str(item.get("claim", "")).strip()
        status = str(item.get("status", "unclear")).strip().lower()
        evidence = str(item.get("evidence", "")).strip()
        if not claim:
            continue
        if status not in allowed:
            status = "unclear"
        quote_found = bool(evidence) and normalize_quote(evidence) in normalized_source
        if status in {"supported", "contradicted"} and not quote_found:
            status, evidence = "manual_review_quote_not_found", ""
        if status in {"not_found", "unclear"}:
            evidence, quote_found = "", False
        results.append({
            "claim": claim, "assessment": status, "source_evidence": evidence,
            "quote_found_in_model_input": bool(quote_found)
        })
    if not results:
        raise ValueError("No usable claims returned. Rerun or inspect the summary manually.")
    return results, tokens, seconds, raw


## 3. Paste a source and run the evaluation

Paste a complete Roman-script Hinglish passage. Inputs longer than 2,048 model tokens are truncated at the end and clearly flagged. Claim checking uses the same model-visible text as summarization.

In [ ]:
source_box = widgets.Textarea(
    value="",
    placeholder="Paste Roman-script Hinglish here, such as Hindi words written in Latin letters mixed with English.",
    description="Source text",
    layout=widgets.Layout(width="100%", height="220px"),
    style={"description_width": "initial"}
)
model_picker = widgets.Dropdown(
    options=list(MODEL_IDS.keys()), value="Qwen3 1.7B",
    description="Model", style={"description_width": "initial"}
)
prompt_picker = widgets.Dropdown(
    options=["Factuality-aware", "Standard"], value="Factuality-aware",
    description="Prompt", style={"description_width": "initial"}
)
run_button = widgets.Button(
    description="Summarize and check claims", button_style="success", icon="play",
    layout=widgets.Layout(width="260px")
)
download_button = widgets.Button(
    description="Download last result (JSON)", icon="download", disabled=True
)
result_output = widgets.Output()

def on_run_clicked(_):
    global _last_result
    run_button.disabled = True
    download_button.disabled = True
    with result_output:
        clear_output(wait=True)
        source = source_box.value.strip()
        if not source:
            display(Markdown("**Paste source text first.**"))
            run_button.disabled = False
            return
        if re.search(r"[\u0900-\u097F]", source):
            display(Markdown("**This demo expects Roman-script Hinglish.** Hindi words should be written in Latin letters, mixed with English."))
            run_button.disabled = False
            return
        if not re.search(r"[A-Za-z]", source):
            display(Markdown("**No Latin-script text found.** Paste a Roman-script Hinglish source."))
            run_button.disabled = False
            return
        try:
            selected_model = model_picker.value
            selected_prompt = prompt_picker.value
            model, tokenizer = load_model(selected_model)
            source_used, original_tokens, source_truncated = prepare_source(source, tokenizer)
            summary, summary_tokens, summary_seconds, summary_cap_hit = make_summary(
                source_used, selected_prompt, model, tokenizer
            )
            if not summary:
                raise ValueError("The model returned an empty summary. Try the other model size.")
            claims, audit_tokens, audit_seconds, raw_audit = check_claims(
                source_used, summary, model, tokenizer
            )
            _last_result = {
                "project": "FactIndic interactive research demo",
                "model": MODEL_IDS[selected_model],
                "prompt": selected_prompt,
                "model_visible_source_sha256": hashlib.sha256(source_used.encode("utf-8")).hexdigest(),
                "source_tokens_before_cap": original_tokens,
                "source_truncated": source_truncated,
                "summary": summary,
                "summary_tokens": summary_tokens,
                "summary_generation_seconds": round(summary_seconds, 3),
                "summary_token_cap_hit": summary_cap_hit,
                "claim_check_tokens": audit_tokens,
                "claim_check_seconds": round(audit_seconds, 3),
                "claims": claims,
                "factuality_status": "AI-assisted, not human-verified",
                "quote_validation": "displayed evidence quotes checked against the model-visible source",
                "all_displayed_quotes_found": all((not c["source_evidence"]) or c["quote_found_in_model_input"] for c in claims)
            }
            display(Markdown("### English summary\n\n" + summary))
            if source_truncated:
                display(Markdown(f"**Input note:** {original_tokens} tokens were provided; the model saw only the first {MAX_SOURCE_TOKENS}."))
            if summary_cap_hit:
                display(Markdown("**Output note:** the summary reached its token limit. Inspect it for an incomplete ending."))
            display(Markdown("### Claim-by-claim evidence\n\nThe quote is checked against the model-visible source. The AI assessment can still be wrong."))
            display(pd.DataFrame(claims))
            display(Markdown(
                f"**Run details:** {selected_model} · {selected_prompt} prompt · "
                f"summary {summary_seconds:.2f}s · claim check {audit_seconds:.2f}s · "
                f"{len(claims)} claims · same model generated and checked the summary."
            ))
            download_button.disabled = False
        except Exception as error:
            _last_result = None
            display(Markdown(f"**Could not complete the check:** `{type(error).__name__}: {error}`. Try a shorter source or the 1.7B model."))
        finally:
            run_button.disabled = False

def on_download_clicked(_):
    if not _last_result:
        return
    output_path = Path("/content/factindic_interactive_result.json")
    output_path.write_text(json.dumps(_last_result, ensure_ascii=False, indent=2), encoding="utf-8")
    files.download(str(output_path))

run_button.on_click(on_run_clicked)
download_button.on_click(on_download_clicked)
display(widgets.VBox([
    source_box, widgets.HBox([model_picker, prompt_picker]),
    widgets.HBox([run_button, download_button]), result_output
]))


## Reading the output

- **Supported** means the AI reviewer found a quote it judged to support the claim.
- **Contradicted** means it found a quote it judged incompatible with the claim.
- **Not found** means it did not locate the claim in the source. This is not proof that the claim is false.
- **Unclear** or **manual review: quote not found** needs a person to inspect the source.

A validated quote only confirms that the text appears in the source; it does not prove the quote logically supports the claim. The same model performs both tasks, so use this as a live project demonstration and AI-assisted review aid.
